In [3]:
import json
import hashlib
import sys
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "gemma4:12b-it-q4_K_M"
API_LOCAL = "http://127.0.0.1:11434"

# Archivos comunes en 05_Pruebas
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
configuracion = json.loads(bytes_config)

# Verificar que no cambiaron
sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == configuracion["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert configuracion["api_url"] == API_LOCAL + "/api/chat"

# Cliente sin proxy hacia Ollama local
cliente_local = build_opener(ProxyHandler({}))

with cliente_local.open(API_LOCAL + "/api/version", timeout=10) as r:
    version_ollama = json.load(r)["version"]

with cliente_local.open(API_LOCAL + "/api/tags", timeout=10) as r:
    modelos = json.load(r)["models"]

modelo_info = next(
    (m for m in modelos if m["name"] == MODELO),
    None
)

assert modelo_info is not None, "Gemma 4 no está instalado."

# Consultar capacidades
peticion = Request(
    API_LOCAL + "/api/show",
    data=json.dumps({"model": MODELO}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with cliente_local.open(peticion, timeout=15) as r:
    detalles = json.load(r)

print("Python:", sys.version.split()[0])
print("Ollama:", version_ollama)
print("Modelo:", MODELO)
print("Identificador:", modelo_info["digest"][:12])
print("Casos verificados:", len(casos))
print("SHA256:", sha_casos)
print("Prompt:", configuracion["prompt_version"])
print("Capacidades:", detalles.get("capabilities"))
print("Thinking:", detalles.get("thinking"))
print("\nArchivos y conexión: correctos")


Python: 3.11.16
Ollama: 0.40.13
Modelo: gemma4:12b-it-q4_K_M
Identificador: 6114515d63c1
Casos verificados: 6
SHA256: f73d159bb43cc9c93d2768e997052f4381a2c24bb046c82826b1793c1b8da37d
Prompt: V3
Capacidades: ['completion', 'vision', 'audio', 'tools', 'thinking']
Thinking: {'values': [False, True], 'default': True}

Archivos y conexión: correctos


In [4]:
import copy
import json
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request

# Diagnóstico sintético, independiente de S-005 a S-010
contenido = (
    "Autores: Autor X y Autor Y.\n"
    "Predicción recibida: favorable (simulada).\n"
    "Modelo predictor y puntuación: no disponibles.\n"
    "[E1] Autor X tiene un artículo sobre bases de datos académicas.\n"
    "[E2] Autor Y tiene un artículo sobre recuperación de información.\n"
    "No se proporcionan datos de colaboración previa."
)

# Copiar la configuración compartida
solicitud = copy.deepcopy(
    configuracion["request_defaults"]
)

solicitud["model"] = MODELO
solicitud["think"] = False

# Configuración inicial de Gemma 4
solicitud["options"].update({
    "num_ctx": 4096,
    "num_predict": 2048,
    "temperature": 1.0,
    "top_p": 0.95,
    "top_k": 64
})

solicitud["messages"] = [
    {
        "role": "system",
        "content": configuracion["system_prompt"]
    },
    {
        "role": "user",
        "content": configuracion["user_prompt_template"].format(
            case_id="DIAG-G4-01",
            contenido=contenido
        )
    }
]

# Evitar guardar diagnósticos duplicados
try:
    with open("resultados.jsonl", encoding="utf-8") as f:
        anteriores = [
            json.loads(line)
            for line in f if line.strip()
        ]
except FileNotFoundError:
    anteriores = []

assert not any(
    r["case_id"] == "DIAG-G4-01" for r in anteriores
), "El diagnóstico ya está registrado."

registro = {
    "execution_id": str(uuid4()),
    "case_id": "DIAG-G4-01",
    "grupo": "diagnostico",
    "configuracion_modelo": "gemma4_directa_v1",
    "modelo_digest": modelo_info["digest"],
    "ollama_version": version_ollama,
    "sha256_casos": sha_casos,
    "sha256_configuracion": sha_config,
    "fecha_inicio_utc": datetime.now(timezone.utc).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

# Solicitud a Ollama local
inicio = time.perf_counter()

try:
    peticion = Request(
        configuracion["api_url"],
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente_local.open(peticion, timeout=360) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"
    elif mensaje.get("thinking"):
        registro["estado"] = "revisar_modo"
    elif (
        respuesta.get("done_reason") == "stop"
        and mensaje.get("content", "").strip()
    ):
        registro["estado"] = "ok"
    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)
registro["fecha_fin_utc"] = datetime.now(timezone.utc).isoformat()

# Guardar el diagnóstico en el archivo común del modelo
with open("resultados.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(registro, ensure_ascii=False) + "\n")

print("ESTADO:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]

    print("Tokens de entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento devuelto:", bool(
        r.get("message", {}).get("thinking")
    ))

    print("\nRESPUESTA:\n")
    print(r.get("message", {}).get("content", ""))
else:
    print("ERROR:", registro.get("error"))

print("\nRegistro guardado en resultados.jsonl")


ESTADO: ok
Tiempo cliente: 45.308 s
Tokens de entrada: 366
Tokens generados: 311
Terminación: stop
Razonamiento devuelto: False

RESPUESTA:

**Evidencia**
[E1] El Autor X cuenta con una publicación sobre bases de datos académicas.
[E2] El Autor Y cuenta con una publicación sobre recuperación de información.
No existen datos documentados sobre colaboraciones previas entre ambos autores.

**Interpretación**
La evidencia muestra que ambos autores trabajan en áreas temáticas relacionadas con la gestión y acceso a la información (bases de datos y recuperación de información). Sin embargo, la evidencia solo respalda la pertinencia académica de una posible colaboración debido a la afinidad de sus campos de estudio, pero no documenta ninguna interacción directa, proyecto compartido o vínculo previo entre el Autor X y el Autor Y.

**Limitaciones**
La evidencia proporcionada es insuficiente para confirmar una relación de coautoría. No hay registros de publicaciones conjuntas ni de filiaciones co

In [5]:
import json
import hashlib
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "gemma4:12b-it-q4_K_M"
API_LOCAL = "http://127.0.0.1:11434"
ARCHIVO = "resultados.jsonl"

# 1. Verificar archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert config["api_url"] == API_LOCAL + "/api/chat"

# 2. Recuperar el diagnóstico exitoso
with open(ARCHIVO, encoding="utf-8") as f:
    anteriores = [json.loads(line) for line in f if line.strip()]

diagnostico = next(
    r for r in anteriores if r["case_id"] == "DIAG-G4-01"
)

assert diagnostico["estado"] == "ok"
assert diagnostico["solicitud"]["model"] == MODELO
assert diagnostico["solicitud"]["think"] is False

opciones = diagnostico["solicitud"]["options"]

assert opciones["num_ctx"] == 4096
assert opciones["num_predict"] == 2048
assert opciones["temperature"] == 1.0
assert opciones["top_p"] == 0.95
assert opciones["top_k"] == 64

assert diagnostico["solicitud"]["messages"][0]["content"] == (
    config["system_prompt"]
)

# 3. Comprobar el modelo instalado
cliente = build_opener(ProxyHandler({}))

with cliente.open(API_LOCAL + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_info["digest"] == diagnostico["modelo_digest"]

# Evitar repetir ejecuciones ya registradas
registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == "gemma4_directa_v1"
    and r.get("grupo") == "comparacion"
}

# 4. Ejecutar los seis casos
with open(ARCHIVO, "a", encoding="utf-8") as archivo:

    for caso in casos:
        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado")
            continue

        # Copiar la solicitud del diagnóstico
        solicitud = copy.deepcopy(
            diagnostico["solicitud"]
        )

        # Cambiar únicamente el contenido del caso
        solicitud["messages"][1]["content"] = (
            config["user_prompt_template"].format(
                case_id=case_id,
                contenido=caso["contenido"]
            )
        )

        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "tipo": caso["tipo"],
            "grupo": "comparacion",
            "configuracion_modelo": "gemma4_directa_v1",
            "prompt_version": "V3",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": diagnostico["ollama_version"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        inicio = time.perf_counter()

        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=360) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            mensaje = respuesta.get("message", {})

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"
            elif mensaje.get("thinking"):
                registro["estado"] = "revisar_modo"
            elif (
                respuesta.get("done_reason") == "stop"
                and mensaje.get("content", "").strip()
            ):
                registro["estado"] = "ok"
            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )

        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # Guardar cada ejecución, incluso si falla
        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        archivo.flush()

        r = registro.get("respuesta_original", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s"
        )

print("\nArchivo actualizado:", ARCHIVO)


S-005 | ok | 402 tokens | 9.967 s
S-006 | ok | 339 tokens | 4.485 s
S-007 | ok | 438 tokens | 4.839 s
S-008 | ok | 359 tokens | 4.292 s
S-009 | ok | 368 tokens | 4.331 s
S-010 | ok | 325 tokens | 3.893 s

Archivo actualizado: resultados.jsonl


In [6]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

ARCHIVO = "resultados.jsonl"
API_LOCAL = "http://127.0.0.1:11434"

# Recuperar el diagnóstico anterior
with open(ARCHIVO, encoding="utf-8") as f:
    registros = [json.loads(line) for line in f if line.strip()]

assert not any(
    r["case_id"] == "DIAG-G4-02" for r in registros
), "Este diagnóstico ya existe."

anterior = next(
    r for r in registros
    if r["case_id"] == "DIAG-G4-01"
)

assert anterior["estado"] == "ok"
assert anterior["solicitud"]["model"] == "gemma4:12b-it-q4_K_M"
assert anterior["solicitud"]["think"] is False

# Conservar prompt y evidencia originales
solicitud = copy.deepcopy(anterior["solicitud"])

# Activar razonamiento, sin aumentar el contexto
solicitud["think"] = True
solicitud["options"]["num_ctx"] = 4096
solicitud["options"]["num_predict"] = 3072

# Verificar que el resto de parámetros no cambió
assert solicitud["options"]["temperature"] == 1.0
assert solicitud["options"]["top_p"] == 0.95
assert solicitud["options"]["top_k"] == 64

registro = {
    "execution_id": str(uuid4()),
    "case_id": "DIAG-G4-02",
    "grupo": "diagnostico",
    "configuracion_modelo": "gemma4_thinking_v1",
    "referencia": "DIAG-G4-01",
    "modelo_digest": anterior["modelo_digest"],
    "ollama_version": anterior["ollama_version"],
    "sha256_casos": anterior["sha256_casos"],
    "sha256_configuracion": anterior["sha256_configuracion"],
    "fecha_inicio_utc": datetime.now(timezone.utc).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

cliente = build_opener(ProxyHandler({}))

inicio = time.perf_counter()

try:
    peticion = Request(
        API_LOCAL + "/api/chat",
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente.open(peticion, timeout=300) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"
    elif (
        respuesta.get("done_reason") == "stop"
        and mensaje.get("content", "").strip()
        and mensaje.get("thinking")
    ):
        registro["estado"] = "ok"
    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)
registro["fecha_fin_utc"] = datetime.now(timezone.utc).isoformat()

# Guardar incluso si ocurre un error
with open(ARCHIVO, "a", encoding="utf-8") as f:
    f.write(json.dumps(registro, ensure_ascii=False) + "\n")

print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        r.get("message", {}).get("thinking")
    ))

    print("\nRESPUESTA FINAL:\n")
    print(r.get("message", {}).get("content", ""))
else:
    print("Error:", registro.get("error"))

print("\nRegistro añadido a resultados.jsonl")


Estado: ok
Tiempo cliente: 15.928 s
Tokens entrada: 364
Tokens generados: 1086
Terminación: stop
Razonamiento separado: True

RESPUESTA FINAL:

**Evidencia**
* [E1] El Autor X posee un artículo sobre bases de datos académicas.
* [E2] El Autor Y posee un artículo sobre recuperación de información.
* No se dispone de datos sobre colaboraciones previas entre ambos autores.

**Interpretación**
La evidencia muestra que ambos autores investigan áreas que están temáticamente vinculadas (bases de datos y recuperación de información) [E1, E2]. Esta coincidencia temática sugiere que ambos autores operan dentro de un marco de investigación común, lo que proporciona una base de relevancia académica para la posible interacción entre sus perfiles.

**Limitaciones**
La evidencia suministrada no demuestra que los autores hayan trabajado juntos anteriormente, ni que compartan un proyecto actual o futuro. La similitud temática entre [E1] y [E2] no es prueba de colaboración. Asimismo, no se proporcionan 

In [7]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "gemma4:12b-it-q4_K_M"
API_LOCAL = "http://127.0.0.1:11434"
ARCHIVO = "resultados.jsonl"

# 1. Leer los archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert config["api_url"] == API_LOCAL + "/api/chat"

# 2. Recuperar resultados existentes
with open(ARCHIVO, encoding="utf-8") as f:
    anteriores = [
        json.loads(line)
        for line in f if line.strip()
    ]

directos = {
    r["case_id"]: r
    for r in anteriores
    if r.get("configuracion_modelo") == "gemma4_directa_v1"
    and r.get("grupo") == "comparacion"
}

assert len(directos) == 6
assert set(directos) == {c["case_id"] for c in casos}

# Diagnóstico con razonamiento
diagnostico = next(
    r for r in anteriores
    if r["case_id"] == "DIAG-G4-02"
)

assert diagnostico["estado"] == "ok"
assert diagnostico["solicitud"]["think"] is True
assert diagnostico["respuesta_original"]["done_reason"] == "stop"

# Recuperar exactamente las opciones utilizadas
opciones_thinking = copy.deepcopy(
    diagnostico["solicitud"]["options"]
)

assert opciones_thinking["num_ctx"] == 4096
assert opciones_thinking["num_predict"] == 3072
assert opciones_thinking["temperature"] == 1.0
assert opciones_thinking["top_p"] == 0.95
assert opciones_thinking["top_k"] == 64

# 3. Comprobar que el modelo no cambió
cliente = build_opener(ProxyHandler({}))

with cliente.open(API_LOCAL + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_info["digest"] == diagnostico["modelo_digest"]

# Evitar ejecuciones duplicadas
registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == "gemma4_thinking_v1"
    and r.get("grupo") == "comparacion"
}

# 4. Ejecutar los seis casos
with open(ARCHIVO, "a", encoding="utf-8") as archivo:

    for caso in casos:

        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado")
            continue

        # Copiar la solicitud directa del mismo caso
        solicitud = copy.deepcopy(
            directos[case_id]["solicitud"]
        )

        # Comprobar que el prompt no cambió
        assert solicitud["messages"][0]["content"] == (
            config["system_prompt"]
        )

        assert solicitud["messages"][1]["content"] == (
            config["user_prompt_template"].format(
                case_id=case_id,
                contenido=caso["contenido"]
            )
        )

        # Aplicar configuración del diagnóstico exitoso
        solicitud["think"] = True
        solicitud["options"] = copy.deepcopy(opciones_thinking)

        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "tipo": caso["tipo"],
            "grupo": "comparacion",
            "configuracion_modelo": "gemma4_thinking_v1",
            "prompt_version": "V3",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": diagnostico["ollama_version"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        inicio = time.perf_counter()

        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=360) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            mensaje = respuesta.get("message", {})

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"

            elif (
                respuesta.get("done_reason") == "stop"
                and mensaje.get("content", "").strip()
                and mensaje.get("thinking")
            ):
                registro["estado"] = "ok"

            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )

        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # Guardar cada resultado, incluso si hay error
        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        archivo.flush()

        r = registro.get("respuesta_original", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s"
        )

print("\nArchivo actualizado:", ARCHIVO)


S-005 | ok | 997 tokens | 13.886 s
S-006 | ok | 1050 tokens | 13.919 s
S-007 | ok | 1457 tokens | 17.613 s
S-008 | ok | 1632 tokens | 20.748 s
S-009 | ok | 1043 tokens | 13.394 s
S-010 | ok | 987 tokens | 11.858 s

Archivo actualizado: resultados.jsonl
